# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Name:', metadata.name)
print('Description:', metadata.description)
print('Published:', getattr(metadata, 'datePublished', 'Unknown'))
print('License:', getattr(metadata, 'license', 'Unknown'))
print('Keywords:', getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We show all record sets in the dataset. For each record set, we enumerate its fields and their `@id`s.

In [ ]:
# List all RecordSet @id's, their fields, and available columns
record_sets = []

print('Record Sets Overview:')
for rs in dataset.record_sets:
    print(f"  RecordSet name: {rs.name}")
    print(f"    @id: {rs.id_}")
    record_sets.append(rs.id_)
    print(f"    Fields:")
    for f in rs.fields:
        field_name = getattr(f, 'name', None)
        print(f"      - {field_name} (@id: {f.id_})")
    print('')
# For the rest of the notebook, use the first RecordSet as an example
if not record_sets:
    raise RuntimeError('No record sets found in the dataset.')
main_record_set = record_sets[0]

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as above.

In [ ]:
# Extract data from each record set into DataFrames
dfs = {}
for recset_id in record_sets:
    recs = list(dataset.records(record_set=recset_id))
    dfs[recset_id] = pd.DataFrame(recs)
    print(f'Record set {recset_id}: {dfs[recset_id].shape[0]} records, columns: {dfs[recset_id].columns.tolist()}')

df = dfs[main_record_set]
print('\nSample of main record set:')
display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

For demonstration, we select a numeric field (e.g., coverage Percentage or MW) and a group field if present.

In [ ]:
# Select a numeric field to analyze
numeric_fields_candidates = ['Coverage', 'Coverage_Percent', 'Coverage (%)', 'MW', 'pI', 'molecular_weight', 'num_peptides', 'num_unique_peptides', 'abundance', 'Normalized_Abundance']
numeric_field = None
for c in numeric_fields_candidates:
    if c in df.columns:
        numeric_field = c
        break
if not numeric_field:
    # Try to find first numeric-looking column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
if not numeric_field:
    raise RuntimeError('No numeric field found in the current DataFrame.')
print(f'Using numeric field for filtering: {numeric_field}')

# Set a filter threshold, e.g. greater than 10
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold} (count: {filtered_df.shape[0]}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a category/label field if one exists
group_candidates = [c for c in df.columns if c.lower() in ['description', 'accession', 'protein', 'label', 'group']]
group_field = group_candidates[0] if group_candidates else None

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    display(grouped_df.head())
else:
    print('No appropriate group field found for grouping.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We show a histogram for the chosen numeric field and, if possible, a boxplot grouped by the categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group if applicable
if group_field:
    plt.figure(figsize=(12,4))
    # Only show top 10 groups for clarity if too many
    top_groups = filtered_df[group_field].value_counts().index[:10].tolist()
    tmp = filtered_df[filtered_df[group_field].isin(top_groups)]
    sns.boxplot(x=group_field, y=numeric_field, data=tmp)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we've:
- Loaded the FAIR^2 Mass Spectrometry dataset from Croissant schema
- Examined metadata, structure, and available record sets
- Loaded the first record set and performed exploratory data analysis
- Filtered, normalized, and visualized a primary numeric field

This workflow can be adapted for further statistical analysis, machine learning, or integration with proteomics pipelines. For details on the columns and fields, always refer to the schema and documentation linked with this dataset.